### PDFs ###

X'/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/Climate_Bond_Framework_of_E1_SUBHOLDING_S_A__in_2021.pdf'

Z'/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/Green_social_bond_framework_series_ad_ae_2019.pdf'

V'/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/ISA_CTM_FRAMEWORK.pdf'

V'/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/Coca-Cola_FEMSA_Green_Bond_Framework.pdf'

~U'/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/SONDA_S_A__GREEN_BOND_FRAMEWORK.pdf' - ~
*moved to a dev NB*

Portuguese

~Y'/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/Green_Bonds_Framework_Athon_2020.pdf' - ~
*moved to a dev NB*

In [1]:
import pymupdf
import spacy
import re
import pandas as pd
import unicodedata
import os
from pathlib import Path

In [2]:
nlp = spacy.load('en_core_web_lg')

In [3]:
# word vectors (similarity) vs linguistic featues e.g. lemma e.g. transportation vs transport

renewable_energy = ['renewable', 'solar', 'wind', 'bioenergy', 'biofuel', 'biomass', 'hydropower', 'hydrogen', 'power', 'grid', 'transmission', 'generation']
energy_efficiency = ['efficiency', 'retrofit']
pollution_prevention_and_control = ['pollution', 'waste']
environmentally_sustainable_management_of_living_natural_resources_and_land_use = ['land', 'agriculture', 'forestry', 'forest', 'fisheries', 'food']
terrestrial_and_aquatic_biodiversity_conservation = ['terrestrial', 'aquatic', 'biodiversity', 'conservation']
clean_transportation = ['transportation', 'electric', 'battery', 'EV', 'charger', 'bus', 'rail', 'train', 'car', 'vehicle', 'bicycle', 'non-motorized', 'aviation']
sustainable_water_and_wastewater_management = ['water', 'potable', 'wastewater', 'sanitation', 'treatment']
climate_change_adaptation = ['adaptation', 'disaster']
circular_economy_and_or_ecoefficient_projects = ['circular', 'recycle', 'reuse']
green_buildings = ['buildings', 'appliances', 'housing']

In [5]:

# document = '/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/Climate_Bond_Framework_of_E1_SUBHOLDING_S_A__in_2021.pdf'

def find_uop(document, language):

    pdf = pymupdf.open(document)

    if language == 'EN':
        keywordsUOP = ['Use of Proceeds', 'Use of Funds', 'Use of the Proceeds']
        keywordsSEEGP = ['Selection and Evaluation', 'Process for the Project Evaluation and Selection', 'Project Evaluation and Selection Process', 'Project Selection and Evaluation Process', 'Process for Project Evaluation and Selection', 'Project Selection and Assessment Process', 'Project Selection Criteria', 'Project evaluation & selection', 'Project Selection Process']

    elif language == 'PT':
        keywordsUOP = ['Uso de Recursos']
        keywordsSEEGP = ['Processo de Avaliação e Seleção de Projetos']

    elif language == 'ES':
        keywordsUOP = ['Uso de fondos', 'Uso de los fondos']
        keywordsSEEGP = ['XYZ']

    areaUOP = None
    areaSEEGP = None

    for page_idx in range(len(pdf)):
        page = pdf[page_idx]

        y1check = []
        y0check = []
        startpage = []
        endpage = []

        if areaUOP is None:
            for keyword in keywordsUOP:
                start = page.search_for(keyword)
                if start:
                    areaUOP = (page_idx, start[0])
                    startpage.append(page_idx)
                    for rect in start:
                        y1check.append(rect.y1)

        if areaSEEGP is None:
            for keyword in keywordsSEEGP:
                end = page.search_for(keyword)
                if end:
                    areaSEEGP = (page_idx, end[0])
                    endpage.append(page_idx)
                    for rect in end:
                        y0check.append(rect.y0)
        
        if len(startpage)>0 and len(endpage)>0:
            if startpage[0] == endpage[0]:
                for point1 in y1check:
                    for point0 in y0check:
                        if (point0 - point1) < 40 and point0 > point1:
                            areaUOP = None
                            areaSEEGP = None

    return areaUOP, areaSEEGP





In [7]:

# iterate pages in the pdf and extract preferrably tables or else words from UOP section

def page_scenario_and_extract(document, areaUOP, areaSEEGP):

    pdf = pymupdf.open(document)
    # language = language

    # inputs
    start_page_idx = areaUOP[0]
    end_page_idx = areaSEEGP[0]
    start_point = areaUOP[1].y1
    end_point = areaSEEGP[1].y0

    # outputs
    noTableMsg = []
    hasTableMsg = []
    hasDFMsg = []
    extractUOPwords = []

# iterate pages
    for page_idx in range(len(pdf)):
        page = pdf[page_idx]

# process four page scenarios, check for tables, extract tables else extract text as words
    # A: UOP all on a single page
        if page_idx == start_page_idx and page_idx == end_page_idx:
            tables = page.find_tables(strategy = 'lines_strict')
            if tables.tables:
                bbox = tables[0].bbox
                if bbox[1] > start_point and bbox[3] < end_point:
                    tableAheader = tables[0].header.names
                    tableAdf = tables[0].to_pandas()
                hasTableMsg.append(tableAheader)
                hasDFMsg.append(tableAdf)
            else:
                noTableMsg.append('No tableA')
                for word in page.get_text('words'):
                    x0, y0, x1, y1, text, *_ = word
                    if y0 > start_point and y1 < end_point:
                        extractUOPwords.append(text)

    # UOP across >1 page
    # B: current page is start page
        elif page_idx == start_page_idx:
            tables = page.find_tables(strategy = 'lines_strict')
            if tables.tables:
                bbox = tables[0].bbox
                if bbox[1] > start_point:
                    tableBheader = tables[0].header.names
                    # tableBheader_cln = " ".join([char for char in tableBheader if char != "\n"])
                    tableBdf = tables[0].to_pandas()
                hasTableMsg.append(tableBheader)
                hasDFMsg.append(tableBdf)
            else:
                noTableMsg.append('No tableB')
                for word in page.get_text('words'):
                    x0, y0, x1, y1, text, *_ = word
                    if y0 > start_point:
                        extractUOPwords.append(text)

    # D: current page is neither start nor end page but is in the UOP area
        elif page_idx > start_page_idx and page_idx < end_page_idx:
            tables = page.find_tables(strategy = 'lines_strict')
            if tables.tables:
                tableDheader = tables[0].header.names
                tableDdf = tables[0].to_pandas()
                hasTableMsg.append(tableDheader)
                hasDFMsg.append(tableDdf)
            else:
                noTableMsg.append(page_idx)
                noTableMsg.append('No tableD')
                for word in page.get_text('words'):
                    x0, y0, x1, y1, text, *_ = word
                    extractUOPwords.append(text)

    # C: current page is end page
        elif page_idx == end_page_idx:
            tables = page.find_tables()
            if tables.tables:
                bbox = tables[0].bbox
                if bbox[3] < end_point:
                    tableCheader = tables[0].header.names
                    tableCdf = tables[0].to_pandas()
                hasTableMsg.append(tableCheader)
                hasDFMsg.append(tableCdf)
            else:
                noTableMsg.append('No tableC')
                for word in page.get_text('words'):
                    x0, y0, x1, y1, text, *_ = word
                    if y1 < end_point:
                        extractUOPwords.append(text)

    return noTableMsg, hasTableMsg, hasDFMsg, extractUOPwords

    # noTableMsg lists page scenarios (and idx for scenario D) where no table is found
    # hasTableMsg lists the header row for found tables
    # hasDFMsg dataframe content in a list





In [8]:
# check extracted table is UOP table and extract Categories

def UOP_table_cats(hasDFMsg, hasTableMsg):

    # inputs
    hasDFMsg = hasDFMsg
    hasTableMsg = hasTableMsg

    # outputs
    uniqueCats = []
    tableInfo = []

    if len(hasDFMsg) >= 1:
        tableInfo.append('More than one table found')
    

    # this over simplifies by assuming the category is always in the first column
    # the header condition code oversimplifies by assuming table fragments split across pages without a header row contain no new category labels

        if 'Category' in hasTableMsg[0]:
            x = 'singular'
        elif 'Categories' in hasTableMsg[0]:
            x = 'plural'
        elif 'Eligible Project Category' in hasTableMsg[0]:
            x = 'phrase'
        else:
            x = 'not a UOP table or category not in 1st columns'

        DFname = hasDFMsg[0]
        if x == 'singular':
            uniqueC = DFname['Category'].unique()
            uniqueCats.append(uniqueC)
        elif x == 'plural':
            uniqueC = DFname['Categories'].unique()
            uniqueCats.append(uniqueC)
        elif x == 'phrase':
            uniqueC = DFname['Eligible Project Category'].unique()
            uniqueCats.append(uniqueC)
        else:
            tableInfo.append(x)

    if len(hasDFMsg) == 0:
        tableInfo.append('No table found')

    return tableInfo, uniqueCats

    # turn these into assert and proper error msgs later
        # if errorMsg is empty and uniqueCats contains a list of category like words, pdf has processed successfully
        # if errorMsg is not empty, there may be more than one table found, in which case DFname variable could be inaccurate
        # if errorMsg is not empty, the words Category or Categories may not be present in the header row, in which case may not be a UOP table or may be other words such as criteria


In [9]:
# process words into category word dataframes where token.similarity score passes threshold

def UOPwords_to_Catwords(extractUOPwords):

    # inputs
    extractUOPwords = extractUOPwords

    re = renewable_energy
    ee = energy_efficiency
    ppc = pollution_prevention_and_control
    esml = environmentally_sustainable_management_of_living_natural_resources_and_land_use
    tabc = terrestrial_and_aquatic_biodiversity_conservation
    ct = clean_transportation
    swwm = sustainable_water_and_wastewater_management
    cca = climate_change_adaptation
    ce = circular_economy_and_or_ecoefficient_projects
    gb = green_buildings

    similarity_threshold = 0.72

    # outputs
    simsre = {}
    simsee = {}
    simsppc = {}
    simsesml = {}
    simstabc = {}
    simsct = {}
    simsswwm = {}
    simscca = {}
    simsce = {}
    simsgb = {}


    # green_buildings
    for wordk in gb:
        dock = nlp(wordk)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if dock.similarity(docb) >= similarity_threshold:
                sim = dock.similarity(docb)
                simsgb[dock[0].text + ' ' + docb[0].text] = sim
                dfgb = pd.DataFrame.from_dict(simsgb, 'index')
            else:
                dfgb = 'is not gb'
    
    
    # renewable_energy
    for worda in re:
        doca = nlp(worda)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if doca.similarity(docb) >= similarity_threshold:
                sim = doca.similarity(docb)
                simsre[doca[0].text + ' ' + docb[0].text] = sim
                dfre = pd.DataFrame.from_dict(simsre, 'index')
            else:
                dfre = 'is not re'


    # energy_efficiency
    for wordc in ee:
        docc = nlp(wordc)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docc.similarity(docb) >= similarity_threshold:
                sim = docc.similarity(docb)
                simsee[docc[0].text + ' ' + docb[0].text] = sim
                dfee = pd.DataFrame.from_dict(simsee, 'index')
            else:
                dfee = 'is not ee'
    
    # pollution_prevention_and_control
    for wordd in ppc:
        docd = nlp(wordd)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docd.similarity(docb) >= similarity_threshold:
                sim = docd.similarity(docb)
                simsppc[docd[0].text + ' ' + docb[0].text] = sim
                dfppc = pd.DataFrame.from_dict(simsppc, 'index')
            else:
                dfppc = 'is not ppc'

    # environmentally_sustainable_management_of_living_natural_resources_and_land_use
    for worde in esml:
        doce = nlp(worde)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if doce.similarity(docb) >= similarity_threshold:
                sim = doce.similarity(docb)
                simsesml[doce[0].text + ' ' + docb[0].text] = sim
                dfesml = pd.DataFrame.from_dict(simsesml, 'index')
            else:
                dfesml = 'is not esml'

    # terrestrial_and_aquatic_biodiversity_conservation
    for wordf in tabc:
        docf = nlp(wordf)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docf.similarity(docb) >= similarity_threshold:
                sim = docf.similarity(docb)
                simstabc[docf[0].text + ' ' + docb[0].text] = sim
                dftabc = pd.DataFrame.from_dict(simstabc, 'index')
            else:
                dftabc = 'is not tabc'

    # clean_transportation
    for wordg in ct:
        docg = nlp(wordg)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docg.similarity(docb) >= similarity_threshold:
                sim = docg.similarity(docb)
                simsct[docg[0].text + ' ' + docb[0].text] = sim
                dfct = pd.DataFrame.from_dict(simsct, 'index')
            else:
                dfct = 'is not ct'

    # sustainable_water_and_wastewater_management
    for wordh in swwm:
        doch = nlp(wordh)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if doch.similarity(docb) >= similarity_threshold:
                sim = doch.similarity(docb)
                simsswwm[doch[0].text + ' ' + docb[0].text] = sim
                dfswwm = pd.DataFrame.from_dict(simsswwm, 'index')
            else:
                dfswwm = 'is not swwm'

    # climate_change_adaptation
    for wordi in cca:
        doci = nlp(wordi)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if doci.similarity(docb) >= similarity_threshold:
                sim = doci.similarity(docb)
                simscca[doci[0].text + ' ' + docb[0].text] = sim
                dfcca = pd.DataFrame.from_dict(simscca, 'index')
            else:
                dfcca = 'is not cca'

    # circular_economy_and_or_ecoefficient_projects
    for wordj in ce:
        docj = nlp(wordj)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docj.similarity(docb) >= similarity_threshold:
                sim = docj.similarity(docb)
                simsce[docj[0].text + ' ' + docb[0].text] = sim
                dfce = pd.DataFrame.from_dict(simsce, 'index')
            else:
                dfce = 'is not ce'

    

    return simsre, simsee, simsppc, simsesml, simstabc, simsct, simsswwm, simscca, simsce, simsgb

In [10]:
# run the program
# notable

document = '/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/refinement_reports/BTG_Pactual_Green,_Social_and_Sustainable_Financing_Framework.pdf'
language = 'EN'
areaUOP, areaSEEGP = find_uop(document, language)
noTableMsg, hasTableMsg, hasDFMsg, extractUOPwords = page_scenario_and_extract(document, areaUOP, areaSEEGP)
tableInfo, uniqueCats = UOP_table_cats(hasDFMsg, hasTableMsg)
if len(uniqueCats) == 0:
    simsre, simsee, simsppc, simsesml, simstabc, simsct, simsswwm, simscca, simsce, simsgb = UOPwords_to_Catwords(extractUOPwords)
filepath = Path(document)

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_58210/2029186089.py:39: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if dock.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_58210/2029186089.py:52: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if doca.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_58210/2029186089.py:65: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if docc.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_58210/2029186089.py:77: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if docd.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_58210/2029186089.py:89: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if doce.similarity(docb) 

In [11]:
print(f"table header {hasTableMsg}")
print('')
print(f"no tables {noTableMsg}")
print('')
print(f" UOP words {extractUOPwords}")
print('')
print(areaUOP)
print(areaSEEGP)

print(uniqueCats)
print(len(hasDFMsg))


table header []

no tables ['No tableB', 13, 'No tableD', 'No tableC']

 UOP words ['Internal', 'Use', 'Only', '13', 'impacts', 'in', 'projects,', 'based', 'on', 'IFC', 'Performance', 'Standards.', 'as', 'BTG', 'Pactual’s', 'Green,', 'Social', 'and', 'Sustainable', 'Financing', 'Framework', 'for', 'issuing', 'debt', 'instruments', 'having', 'environmental', 'and/or', 'social', 'impact.', 'Category', 'A', 'refers', 'to', 'high', 'environmental', 'and', 'social', 'risk', 'for', 'the', 'project,', 'due', 'to', 'the', 'potential', 'risk', 'of', 'activity', 'giving', 'rise', 'to', 'significant', 'adverse', 'environmental', 'or', 'social', 'impacts', 'that', 'are', 'varied,', 'irreversible', 'or', 'unprecedented', 'o', 'Risk', 'management', 'framework', 'for', 'determining,', 'assessing', 'and', 'managing', 'E&S', 'risks', 'Risk', 'Management', 'Land', 'Resettlement', 'Labor', 'Biodiversity', 'Resource', 'Efficiency', 'Indigenous', 'People', 'Community', 'Cultural', 'Heritage', 'o', 'IFC', '

In [12]:
# collate the outputs
# notable

print(filepath.name, type(filepath.name))
print(language, type(language))
print(areaUOP[0], type(areaUOP[0]))
print(areaSEEGP[0], type(areaSEEGP[0]))
if len(hasDFMsg) != 0:
    if len(tableInfo) == 1:   
        print(tableInfo[0], type(tableInfo[0]))
        if len(uniqueCats) > 0:
            print(uniqueCats, len(uniqueCats))
    elif len(tableInfo) == 2:
        print(tableInfo[1], type(tableInfo[1]))
        if len(uniqueCats) > 0:
            print(uniqueCats, len(uniqueCats))
    else:
        print('no categories from tables')
if len(hasDFMsg) == 0:
    print(f"renewable_energy  {simsre}")
    print(f"energy_efficiency {simsee}")
    print(f"pollution_prevention_and_control {simsppc}")
    print(f"environmentally_sustainable_management_of_living_natural_resources_and_land_use {simsesml}")
    print(f"terrestrial_and_aquatic_biodiversity_conservation {simstabc}")
    print(f"clean_transportation {simsct}")
    print(f"sustainable_water_and_wastewater_management {simsswwm}")
    print(f"climate_change_adaptation {simscca}")
    print(f"circular_economy_and_or_ecoefficient_projects {simsce}")
    print(f"green_buildings {simsgb}")
else:
    print('categories from tables')

BTG_Pactual_Green,_Social_and_Sustainable_Financing_Framework.pdf <class 'str'>
EN <class 'str'>
12 <class 'int'>
14 <class 'int'>
renewable_energy  {}
energy_efficiency {'efficiency Efficiency': 1.0000001192092896}
pollution_prevention_and_control {'pollution environmental': 0.7345117926597595}
environmentally_sustainable_management_of_living_natural_resources_and_land_use {'land Land': 1.0000001192092896}
terrestrial_and_aquatic_biodiversity_conservation {'biodiversity Biodiversity': 1.0, 'conservation Biodiversity': 0.748937726020813}
clean_transportation {}
sustainable_water_and_wastewater_management {}
climate_change_adaptation {}
circular_economy_and_or_ecoefficient_projects {}
green_buildings {}


In [229]:
tableInfo

['More than one table found']